In [2]:
# Cell 1: Setup (Gemini planner pipeline, independent)
import os
import re
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from google import genai
from google.genai import types

# Project root (run notebook from formalizer-scaling/src/planner_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Data / prompt / output paths
DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "llm_planner_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"

# New output root for direct LLM-as-planner pipeline
MODEL_DIRNAME = "gemini_3_flash"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner" / MODEL_DIRNAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# VAL executable (adjust if your environment differs)
VAL_PATH = Path.home() / "tools" / "VAL" / "build" / "linux64" / "Release" / "bin" / "Validate"

# Model + API
MODEL_NAME = "gemini-3-flash-preview"
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError(f"GOOGLE_API_KEY not found in {ENV_PATH}")

client = genai.Client(api_key=GOOGLE_API_KEY)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/llm_planner/gemini_3_flash
VAL exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [3]:
# Cell 2: Load problem IDs, planner instruction, and ground-truth domain PDDL
import pandas as pd

# Use all available problems by default; edit this list later for targeted runs
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 201)]

PLANNER_INSTRUCTION = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Planner prompt chars:", len(PLANNER_INSTRUCTION))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 200
First/last: p01 p200
Planner prompt chars: 1294
Domain PDDL chars: 997


In [4]:
# Cell 3: Helper - load NL domain/problem descriptions for one problem ID
def load_descriptions(problem_id: str):
  domain_desc_path = DATA_DIR / "descriptions" / f"{problem_id}_domain.txt"
  problem_desc_path = DATA_DIR / "descriptions" / f"{problem_id}_problem.txt"

  if not domain_desc_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_desc_path}")
  if not problem_desc_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_desc_path}")

  return domain_desc_path.read_text(), problem_desc_path.read_text()

In [5]:
# Cell 4: Helper - build planner prompt (NL domain + NL problem + GT domain PDDL)
def build_prompt(problem_id: str):
  domain_desc, problem_desc = load_descriptions(problem_id)

  return (
      PLANNER_INSTRUCTION
      + "\n\nYou are an expert automated planner.\n"
      "Create an executable plan that is consistent with all provided information.\n\n"
      "### Natural-language domain description\n"
      f"{domain_desc}\n\n"
      "### Natural-language problem description\n"
      f"{problem_desc}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output only the final plan wrapped in <plan>...</plan>."
  )

In [54]:
# Cell 5: Helper - call Gemini for direct plan generation
def call_gemini(prompt: str):
  resp = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config=types.GenerateContentConfig(
          temperature=0.4,
          max_output_tokens=180000,
      ),
  )
  return resp.text or ""

In [7]:
# Cell 6: Helper - extract <plan>...</plan> block
def extract_plan(raw_text: str):
  if not raw_text:
      return ""

  m = re.search(r"<plan>(.*?)</plan>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if not m:
      return ""

  plan_text = m.group(1).strip()

  # Keep one action per non-empty line
  lines = [ln.strip() for ln in plan_text.splitlines() if ln.strip()]
  return "\n".join(lines)

In [8]:
# Cell 7: Helper - validate a plan with VAL (GT domain + GT problem)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  proc = subprocess.run(cmd, capture_output=True, text=True)

  out_text = (proc.stdout or "").strip()
  err_text = (proc.stderr or "").strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [9]:
# Cell 8: Helper - run one problem end-to-end (generate plan + VAL), persist outputs
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_gemini(prompt)
  plan_text = extract_plan(raw_text)

  cand_dir = OUTPUT_ROOT / problem_id / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  raw_path = cand_dir / "raw.txt"
  plan_path = cand_dir / "plan.txt"
  val_log_path = cand_dir / "validator.log"
  err_path = cand_dir / "planner_error.txt"

  raw_path.write_text(raw_text or "")

  if not plan_text:
      plan_path.write_text("")
      err_msg = "Empty or missing <plan>...</plan> output."
      err_path.write_text(err_msg)
      return {
          "problem": problem_id,
          "plan_found": False,
          "val_ok": False,
          "planner_error": err_msg,
      }

  plan_path.write_text(plan_text + ("" if plan_text.endswith("\n") else "\n"))

  gt_problem_path = DATA_DIR / "pddl" / f"{problem_id}.pddl"
  if not gt_problem_path.exists():
      err_msg = f"Missing ground-truth problem file: {gt_problem_path}"
      err_path.write_text(err_msg)
      return {
          "problem": problem_id,
          "plan_found": True,
          "val_ok": False,
          "planner_error": err_msg,
      }

  val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)
  val_log_path.write_text(val_result.get("val_output", "") or "")

  return {
      "problem": problem_id,
      "plan_found": True,
      "val_ok": bool(val_result["val_ok"]),
      "planner_error": "",
  }

In [73]:
# Cell 9: Generate only (no VAL)
smoke_problem = "p99"

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
plan_text = extract_plan(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "plan.txt").write_text((plan_text + "\n") if plan_text else "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_plan_chars:", len(plan_text or ""))
print("saved:", cand_dir)

Generation-only done.
problem: p99
extracted_plan_chars: 1776
saved: /home/sagemaker-user/formalizer-scaling/output/llm_planner/gemini_3_flash/p99/cand_00


In [74]:
# Cell 10: VAL only (no model call), concise output
import contextlib
import io

smoke_problem = "p99"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

plan_path = cand_dir / "plan.txt"
if not plan_path.exists():
  raise FileNotFoundError(f"Missing plan file: {plan_path}")

plan_text = plan_path.read_text().strip()
if not plan_text:
  summary = {
      "problem": smoke_problem,
      "plan_found": False,
      "val_ok": False,
      "planner_error": "Empty plan.txt (no actions).",
  }
  (cand_dir / "planner_error.txt").write_text(summary["planner_error"])
  print("Produced files:")
  print("- planner_error.txt")
  print("VAL result: not run (empty plan)")
  print("Summary:")
  print(json.dumps(summary, indent=2))
else:
  gt_problem_path = DATA_DIR / "pddl" / f"{smoke_problem}.pddl"
  if not gt_problem_path.exists():
      raise FileNotFoundError(f"Missing ground-truth problem file for VAL: {gt_problem_path}")

  with contextlib.redirect_stdout(io.StringIO()):
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_path)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  summary = {
      "problem": smoke_problem,
      "plan_found": True,
      "val_ok": bool(val_result["val_ok"]),
      "planner_error": "",
  }

  print("Produced files:")
  print("- validator.log")
  print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
  print("Summary:")
  print(json.dumps(summary, indent=2))

Produced files:
- validator.log
VAL result: invalid
Summary:
{
  "problem": "p99",
  "plan_found": true,
  "val_ok": false,
  "planner_error": ""
}
